# 🎧 NexTune — Bluetooth Headphones Price Prediction
### Model Training & Evaluation

**Problem Statement:** Your company is planning to release a new wireless Bluetooth headphone in the Indian market. Recommend a suitable price based on the specifications and market demand.

---
> **▶️ HOW TO RUN:** Open in [Google Colab](https://colab.research.google.com) → Runtime → Run all

## Step 1: Load the Dataset

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Load dataset
URL = 'https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/final-merged-dataset.csv' \
      if os.path.exists('/content') else '../data/final-merged-dataset.csv'

df = pd.read_csv(URL)
print(f"Dataset shape : {df.shape}")
print(f"Columns       : {df.columns.tolist()}")
df.head()

## Step 2: Data Preprocessing

Fix data types, handle missing values, engineer features, encode categoricals.

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Fix numeric types
num_cols = ['price_inr','rating','review_count','battery_life_hrs','bluetooth_version',
            'driver_size_mm','ipx_level','latency_ms','anc_db']
bin_cols = ['has_noise_cancellation','has_enc','has_usb_c','has_premium_codec',
            'has_touch_control','has_voice_assistant','has_fast_charge','has_dual_pairing',
            'has_gaming_mode','has_hi_res_audio','has_spatial_audio','has_low_latency']

for c in num_cols + bin_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
for c in bin_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

# Drop rows with missing price or brand
df = df.dropna(subset=['price_inr','brand'])
df = df[df['price_inr'] > 0].reset_index(drop=True)

# Log-transform price (reduces MAPE impact of very low-priced products)
df['log_price'] = np.log1p(df['price_inr'])

# Brand tier
brand_avg = df.groupby('brand')['price_inr'].mean()
df['brand_tier'] = df['brand'].apply(
    lambda b: 'premium' if brand_avg.get(b,0)>=10000 else ('mid' if brand_avg.get(b,0)>=3000 else 'budget')
)

# Encode categoricals
le_dict = {}
for col in ['category','brand','brand_tier','country_of_origin']:
    if col in df.columns:
        le = LabelEncoder()
        df[col+'_enc'] = le.fit_transform(df[col].fillna('Unknown').astype(str))
        le_dict[col] = le

# Engineered features
df['has_anc']        = df['has_noise_cancellation'].fillna(0).astype(int)
df['bt_major']       = df['bluetooth_version'].apply(lambda x: int(x) if pd.notna(x) else 5)
df['high_rating']    = (df['rating'] >= 4.0).astype(int)
df['price_per_hour'] = df['price_inr'] / (df['battery_life_hrs'].fillna(0) + 1)

print(f"Clean dataset : {len(df)} products")
print(f"Price range   : ₹{df.price_inr.min():.0f} – ₹{df.price_inr.max():.0f}")
print(f"Median price  : ₹{df.price_inr.median():.0f}")
df[['price_inr','rating','battery_life_hrs','bluetooth_version','has_anc','brand_tier']].describe().round(2)

## Step 3: Feature Selection

Only features with **positive correlation** to price are kept. Negative-correlation features add noise and hurt accuracy.

In [ ]:
# Features with POSITIVE correlation to price (verified from EDA)
FEATURES = [
    'brand_tier_enc',    # r = +0.776  ← strongest signal
    'price_per_hour',    # r = +0.919  ← value metric
    'has_anc',           # r = +0.392
    'rating',            # r = +0.205
    'has_hi_res_audio',  # r = +0.199
    'high_rating',       # r = +0.174
    'has_spatial_audio', # r = +0.165
    'bluetooth_version', # r = +0.064
    'brand_enc',         # r = +0.053
    'has_dual_pairing',  # r = +0.052
    'has_premium_codec', # r = +0.038
    'bt_major',          # r = +0.036
    'anc_db',            # r = +0.026
    'has_enc',           # r = +0.026
    'review_count',      # r = +0.025
    'ipx_level',         # r = +0.016
    'driver_size_mm',    # r = +0.016
    'has_low_latency',   # r = +0.003
]
# DROPPED (negative correlation): category_enc, has_fast_charge, has_touch_control,
# has_ipx, has_voice_assistant, has_gaming_mode, has_usb_c, battery_life_hrs, latency_ms

FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'log_price'

X = df[FEATURES].fillna(df[FEATURES].median())
y = df[TARGET]

print(f"Feature matrix : {X.shape}")
print(f"Features used  : {FEATURES}")

## Step 4: Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Keep original price for evaluation
_, _, y_train_orig, y_test_orig = train_test_split(
    X, df['price_inr'], test_size=0.2, random_state=42
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set : {len(X_train)} samples  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test set     : {len(X_test)} samples  ({len(X_test)/len(X)*100:.0f}%)")
print(f"Train price  : ₹{y_train_orig.min():.0f} – ₹{y_train_orig.max():.0f}")
print(f"Test price   : ₹{y_test_orig.min():.0f} – ₹{y_test_orig.max():.0f}")

## Step 5: Train Multiple Models

> Models are trained on **log(price)** to reduce MAPE. Predictions are converted back to ₹ using `exp()`.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

models = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(alpha=10.0),
    'Lasso'             : Lasso(alpha=10.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42),
}

results      = {}
predictions  = {}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred_log = model.predict(X_test_sc)
    y_pred     = np.expm1(y_pred_log)          # back to ₹
    y_true     = y_test_orig.values

    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    cv   = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2').mean()

    results[name]     = {'R²':r2, 'CV R²':cv, 'RMSE':rmse, 'MAE':mae, 'MAPE%':mape}
    predictions[name] = y_pred

results_df = pd.DataFrame(results).T.sort_values('R²', ascending=False).reset_index()
results_df.columns = ['Model','R²','CV R²','RMSE','MAE','MAPE%']
print(results_df.round(4).to_string(index=False))

## Step 6: Model Evaluation — Metrics Summary

In [ ]:
print("=" * 70)
print(f"  {'Model':<22}  {'R²':>6}  {'CV R²':>6}  {'RMSE':>8}  {'MAE':>7}  {'MAPE':>7}")
print(f"  {'-'*22}  {'-'*6}  {'-'*6}  {'-'*8}  {'-'*7}  {'-'*7}")
for _, row in results_df.iterrows():
    flag = ' ✅' if row['MAPE%'] < 20 else ''
    print(f"  {row['Model']:<22}  {row['R²']:>6.4f}  {row['CV R²']:>6.4f}  "
          f"₹{row['RMSE']:>6.0f}  ₹{row['MAE']:>5.0f}  {row['MAPE%']:>5.1f}%{flag}")
print("=" * 70)

best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
print(f"\n🏆 Best Model : {best_name}")
print(f"   R²          : {results_df.iloc[0]['R²']:.4f}")
print(f"   CV R²       : {results_df.iloc[0]['CV R²']:.4f}")
print(f"   RMSE        : ₹{results_df.iloc[0]['RMSE']:.2f}")
print(f"   MAE         : ₹{results_df.iloc[0]['MAE']:.2f}")
print(f"   MAPE        : {results_df.iloc[0]['MAPE%']:.2f}%")

## Step 7: Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R² comparison
colors = ['#2ecc71' if r >= 0.70 else '#e74c3c' for r in results_df['R²']]
axes[0].barh(results_df['Model'], results_df['R²'], color=colors)
axes[0].axvline(0.70, color='black', ls='--', lw=1.5, label='Target R²=0.70')
axes[0].set_title('R² Score (higher = better)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('R²'); axes[0].legend()
for i,v in enumerate(results_df['R²']): axes[0].text(v+0.005, i, f'{v:.3f}', va='center', fontsize=9)

# RMSE
axes[1].barh(results_df['Model'], results_df['RMSE'], color='#378ADD')
axes[1].set_title('RMSE ₹ (lower = better)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('RMSE (INR)')
for i,v in enumerate(results_df['RMSE']): axes[1].text(v+5, i, f'₹{v:.0f}', va='center', fontsize=9)

# MAPE
colors_m = ['#2ecc71' if v < 20 else '#e74c3c' for v in results_df['MAPE%']]
axes[2].barh(results_df['Model'], results_df['MAPE%'], color=colors_m)
axes[2].axvline(20, color='black', ls='--', lw=1.5, label='Target MAPE=20%')
axes[2].set_title('MAPE% (lower = better)', fontweight='bold', fontsize=12)
axes[2].set_xlabel('MAPE %'); axes[2].legend()
for i,v in enumerate(results_df['MAPE%']): axes[2].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Model Comparison — NexTune Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Actual vs Predicted + Residuals for best model
y_pred_best = predictions[best_name]
y_true      = y_test_orig.values
residuals   = y_true - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

lims = [min(y_true.min(), y_pred_best.min()), max(y_true.max(), y_pred_best.max())]
axes[0].scatter(y_true, y_pred_best, alpha=0.6, color='#378ADD', edgecolors='white', s=70)
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (₹)', fontsize=12)
axes[0].set_ylabel('Predicted Price (₹)', fontsize=12)
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontweight='bold', fontsize=13)
axes[0].legend()

axes[1].scatter(y_pred_best, residuals, alpha=0.5, color='#1D9E75', s=60)
axes[1].axhline(0, color='red', ls='--', lw=2)
axes[1].set_xlabel('Predicted Price (₹)', fontsize=12)
axes[1].set_ylabel('Residual (Actual − Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontweight='bold', fontsize=13)

plt.tight_layout(); plt.show()
print(f"Mean residual : ₹{residuals.mean():.0f}  (close to 0 = unbiased)")
print(f"Std residual  : ₹{residuals.std():.0f}")

## Step 8: Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp   = best_model.feature_importances_
    label = 'Feature Importance'
else:
    imp   = np.abs(best_model.coef_)
    label = '|Coefficient|'

feat_imp = pd.Series(imp, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
colors = ['#2ecc71' if v >= feat_imp.median() else '#378ADD' for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors)
plt.title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel(label)
plt.tight_layout(); plt.show()

print("Top 10 most important features:")
print(feat_imp.sort_values(ascending=False).head(10).round(4).to_string())

## Step 9: Save Model Artifacts

In [ ]:
import os

SAVE_DIR = '/content/models' if os.path.exists('/content') else '../models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_model, f'{SAVE_DIR}/price_predictor.pkl')
joblib.dump(scaler,     f'{SAVE_DIR}/scaler.pkl')
joblib.dump(le_dict,    f'{SAVE_DIR}/label_encoders.pkl')
joblib.dump(FEATURES,   f'{SAVE_DIR}/features.pkl')
joblib.dump(brand_avg.to_dict(), f'{SAVE_DIR}/brand_avg.pkl')

print(f"✅ Saved to {SAVE_DIR}/")
print(f"   price_predictor.pkl  — {best_name} model")
print(f"   scaler.pkl           — StandardScaler")
print(f"   label_encoders.pkl   — LabelEncoders for brand/category/tier")
print(f"   features.pkl         — feature list ({len(FEATURES)} features)")
print(f"   brand_avg.pkl        — brand average prices for tier calculation")

if os.path.exists('/content'):
    from google.colab import files
    for f in ['price_predictor.pkl','scaler.pkl','label_encoders.pkl','features.pkl','brand_avg.pkl']:
        files.download(f'{SAVE_DIR}/{f}')

## Step 10: Predict Price for a New Headphone

Simulate recommending a price for a new product your company is about to launch.

In [ ]:
def predict_price(brand, has_anc, is_tws, is_neckband=False,
                  bt_version=5.3, driver_mm=10, rating=4.0, reviews=0,
                  has_hi_res=0, has_spatial=0, has_dual=0,
                  has_codec=0, has_enc=0, has_low_lat=0,
                  ipx_level=0, anc_db=0):

    tier = 'premium' if brand_avg.get(brand,0)>=10000 else ('mid' if brand_avg.get(brand,0)>=3000 else 'budget')

    def enc(col, val):
        le = le_dict.get(col)
        return int(le.transform([val])[0]) if le and val in le.classes_ else 0

    row = {
        'brand_tier_enc':    enc('brand_tier', tier),
        'price_per_hour':    0,
        'has_anc':           int(has_anc),
        'rating':            float(rating),
        'has_hi_res_audio':  int(has_hi_res),
        'high_rating':       int(float(rating) >= 4.0),
        'has_spatial_audio': int(has_spatial),
        'bluetooth_version': float(bt_version),
        'brand_enc':         enc('brand', brand),
        'has_dual_pairing':  int(has_dual),
        'has_premium_codec': int(has_codec),
        'bt_major':          int(float(bt_version)),
        'anc_db':            float(anc_db),
        'has_enc':           int(has_enc),
        'review_count':      float(reviews),
        'ipx_level':         float(ipx_level),
        'driver_size_mm':    float(driver_mm),
        'has_low_latency':   int(has_low_lat),
    }

    vec      = np.array([[row.get(f, 0) for f in FEATURES]])
    vec_sc   = scaler.transform(vec)
    log_pred = best_model.predict(vec_sc)[0]
    price    = float(np.expm1(log_pred))

    category = 'TWS Earbuds' if is_tws else ('Neckband' if is_neckband else 'Over-Ear')

    print("\n" + "=" * 55)
    print("       💡  PRICE RECOMMENDATION")
    print("=" * 55)
    print(f"  Brand          : {brand}  ({tier} tier)")
    print(f"  Category       : {category}")
    print(f"  ANC            : {'Yes' if has_anc else 'No'}")
    print(f"  Bluetooth      : v{bt_version}")
    print(f"  Driver         : {driver_mm}mm")
    print(f"  Rating         : {rating}")
    print(f"  IPX Level      : {'IPX'+str(ipx_level) if ipx_level else 'None'}")
    print("-" * 55)
    print(f"  Predicted      : ₹{price:,.0f}")
    print(f"  Suggested Range: ₹{price*0.9:,.0f} – ₹{price*1.1:,.0f}  (±10%)")
    print("=" * 55)


# ✏️ Edit these specs for your new headphone
predict_price(
    brand='Boat',
    has_anc=True,
    is_tws=True,
    bt_version=5.3,
    driver_mm=10,
    rating=4.2,
    reviews=0,
    has_hi_res=0,
    has_spatial=0,
    has_dual=0,
    has_codec=0,
    has_enc=1,
    has_low_lat=1,
    ipx_level=4,
    anc_db=30
)